In [2]:
import os
import numpy as np

midi_file_paths = []
for root, dirs, files in os.walk("POP909-Dataset/POP909"):
    for file in files:
        if file.endswith(".mid") and not root.endswith("versions"):
            midi_file_paths.append(os.path.join(root, file))

print(f"\nTotal MIDI files found: {len(midi_file_paths)}")


Total MIDI files found: 909


In [3]:
import mido

def ExtractMelodyTones(file_path):

    abs_path = file_path

    midi_mel = mido.MidiFile(abs_path)
    seq = []

    for msg in midi_mel.tracks[1]: #Vi vil kun have melodistemen
        if msg.type == 'note_on' and msg.velocity > 0:  # Kun note_on with velocity > 0
            if seq == []:
                seq.append(msg.note)
            elif msg.note != seq[-1]:
                seq.append(msg.note)

    return seq

def ConvertToFreqs(seq):
    return [round(440 * (2**((tone - 69)/12)), 3) for tone in seq]

melodies_midi = [ExtractMelodyTones(path) for path in midi_file_paths]
melodies_freqs = [ConvertToFreqs(seq) for seq in melodies_midi]

In [4]:
print("There is " + str(len(melodies_freqs)) + " sequences")
print("with an average length of " + str(sum([len(seq) for seq in melodies_freqs]) / len(melodies_freqs)) + " tones")

individual_tones = set()
for seq in melodies_midi:
    for tone in seq:
        individual_tones.add(tone)

print("Consisting of " + str(len(individual_tones)) + " unique tones")

There is 909 sequences
with an average length of 260.1771177117712 tones
Consisting of 55 unique tones


### Variable Order Markov Model

First we need the n-grams the maximum order of seven (septogram)

In [5]:
n_grams = []
max_order = 7

for i in range(max_order+1):
    i_gram = []
    if i == 0:
        for seq in melodies_midi:
            for tone in seq:
                i_gram.append(tuple([tone]))
    else:
        for seq in melodies_midi:
            for j, tone in enumerate(seq[:-i]):
                i_gram.append(tuple(seq[j:j+i+1]))

    n_grams.append(i_gram)



In [6]:
for i, n_gram in enumerate(n_grams):
    print("There is " + str(len(n_gram)) + " " + str(i) + "-grams in the corpus")

There is 236501 0-grams in the corpus
There is 235592 1-grams in the corpus
There is 234683 2-grams in the corpus
There is 233774 3-grams in the corpus
There is 232865 4-grams in the corpus
There is 231956 5-grams in the corpus
There is 231047 6-grams in the corpus
There is 230138 7-grams in the corpus


In [7]:
from collections import defaultdict, Counter

n_gram_counts = [defaultdict(int) for _ in range(8)]

for order, i_grams in enumerate(n_grams):
    for gram in i_grams:
        n_gram_counts[order][gram] += 1 

Calculates the probabilities

$$
p_{esc}=number of distinct next symbols/total count for this context​
$$

In [157]:
def PredictNext(seq):
    probs = defaultdict(float)
    remaining_prob = 1.0
    order = min(len(seq), max_order)
    num_tones = 5

    for o in range(order, 0, -1):
        context = seq[order - o :]
        total = sum(
            n_gram_counts[o][context + (s,)] for s in individual_tones
        )
        if total == 0:
            continue
        distinct = sum(
            1 for s in individual_tones if n_gram_counts[o][context + (s,)] > 0
        )
        p_esc = distinct / (total + distinct)

        for s in individual_tones:
            count = n_gram_counts[o][context + (s,)]
            if count > 0:
                probs[s] += remaining_prob * (1 - p_esc) * (count / total)
        
        remaining_prob *= p_esc

    unseen_prob = remaining_prob / len(individual_tones)
    for s in individual_tones:
        probs[s] += unseen_prob
    
    probs.pop(seq[-1], None)
    
    return dict(sorted(probs.items(), key=lambda x: -x[1])[:num_tones])


Generating a simple melodi with pure probability

In [158]:
def ToneRandChoice(probs):
    total = sum(probs.values())
    choice = np.random.random() * total
    for key, value in probs.items():
        if choice < value:
            return key, value
        choice -= value


def GenerateMelody(probe, length):
    context = (probe,)
    all_probs = []
    for i in range(length):
        pred = PredictNext(context)
        new_tone, prob = ToneRandChoice(pred)
        all_probs.append(prob)
        context += (new_tone,)
    return list(context), all_probs 


Using this to create binary trees for sequential melody production
$$
N=2^0+2^1+...+2^{l-1}=2^l-1
$$

In [15]:
import math

def GenerateBinaryTree(probe, length):
    tree = [0 for _ in range(2**length-1)]
    prob_tree = [0 for _ in range(2**length-1)]
    tree[0] = probe
    prob_tree[0] = 1
    for layer in range(1,length,1):
        fst_pos = 2**layer - 1
        for node in range(0,2**layer, 2):
            parents = [math.ceil((node+fst_pos)/2)-1]
            for i in range(layer-2):
                parents.append(math.ceil(parents[-1]/2) - 1)
            context = tuple([tree[i] for i in parents])
            pred = PredictNext(context)
            tone1, prob1 = ToneRandChoice(pred)
            pred.pop(tone1)
            tone2, prob2 = ToneRandChoice(pred)
            choice = np.random.random()
            if choice > 0.5:
                tone1, tone2 = tone2, tone1
                prob1, prob2 = prob2, prob1
            tree[fst_pos + node] = tone1
            tree[fst_pos + node + 1] = tone2

            prob_tree[fst_pos + node] = prob1
            prob_tree[fst_pos + node + 1] = prob2
    return tree, prob_tree

tones, probs = GenerateBinaryTree(67,5)     
print(tones)
print(probs)

[67, 69, 65, 71, 66, 67, 68, 66, 71, 71, 73, 67, 63, 70, 63, 71, 66, 73, 68, 68, 71, 71, 66, 67, 64, 67, 63, 68, 63, 61, 70]
[1, 0.2655477557657546, 0.21524032144490504, 0.24752550202091486, 0.07674215692564317, 0.30048255588218986, 0.09657773586998726, 0.11263914402761112, 0.34767047461408307, 0.4123727094283082, 0.05039214798377396, 0.3708276359011298, 0.23690632385841454, 0.10236264803284977, 0.2641772185914738, 0.2918473769102481, 0.23336480985803165, 0.033812556498156746, 0.06463335211809948, 0.06036193060803286, 0.6578023746136153, 0.4738089668532641, 0.356620871570127, 0.3708276359011298, 0.05490401651418405, 0.27382682286821514, 0.4584833377101283, 0.3910506995524627, 0.27244006160643186, 0.2756528005315063, 0.16702464364134087]


Just for testing the tree, i will make a function to choose a random sequence

In [205]:
def TreeRandSeq(tree, length):
    seq = [tree[0]]
    parent = 0
    for i in range(length-1):
        choice = np.random.random()
        if choice > 0.5:
            seq.append(tree[2*(parent+1)-1])
            parent = 2*(parent+1)-1
        else:
            seq.append(tree[2*(parent+1)])
            parent = 2*(parent+1)
    return seq

SomeTree = ConvertToFreqs(GenerateBinaryTree(77,3))
print(SomeTree)
print(TreeRandSeq(SomeTree, 3))

[698.456, 830.609, 587.33, 739.989, 698.456, 698.456, 493.883]
[698.456, 830.609, 739.989]


Generating a melody with change
- Input is probe, position and suprise-factor

In [159]:
import random

def PredictNextWithAlternative(seq, surprise):
    probs = defaultdict(float)
    remaining_prob = 1.0
    order = len(seq)
    num_tones = 5

    for o in range(order, 0, -1):
        context = seq[order - o :]
        total = sum(
            n_gram_counts[o][context + (s,)] for s in individual_tones
        )
        if total == 0:
            continue
        distinct = sum(
            1 for s in individual_tones if n_gram_counts[o][context + (s,)] > 0
        )
        p_esc = distinct / (total + distinct)

        for s in individual_tones:
            count = n_gram_counts[o][context + (s,)]
            if count > 0:
                probs[s] += remaining_prob * (1 - p_esc) * (count / total)
        
        remaining_prob *= p_esc

    unseen_prob = remaining_prob / len(individual_tones)
    for s in individual_tones:
        probs[s] += unseen_prob

    probs.pop(seq[-1], None)

    surprisal1, surprisal2 = surprise

    if surprisal1:
        min_val1, max_val1 = 0.01, 0.05 
    else:
        min_val1, max_val1 = 0.15, 1
    
    if surprisal2:
        min_val2, max_val2 = 0.01, 0.05 
    else:
        min_val2, max_val2 = 0.15, 1

    keys = list(probs.keys())
    values = list(probs.values())

    picked_real, picked_alt = False, False

    real, alt = False, False
    i,j = 0,0

    while not picked_real and i < 100:
        selected_key = random.choices(keys, weights=values, k=1)[0]
        selected_value = probs[selected_key]

        if min_val1 <= selected_value <= max_val1:
            real = selected_key, selected_value
            picked_real = True
        
        i += 1
    
    while not picked_alt and j < 100:
        selected_key = random.choices(keys, weights=values, k=1)[0]
        selected_value = probs[selected_key]

        if min_val2 <= selected_value <= max_val2:
            alt = selected_key, selected_value
            picked_alt = True

        j += 1
    
    return real, alt


def GenerateMelodyWithChange(probe, length, pos, surprise):
    context = (probe,)
    probs = []
    for i in range(length-1):
        if i+1 == pos:
            tone, alt = PredictNextWithAlternative(context, surprise)
            if tone == False or alt == False:
                return False
            new_tone, val = tone
        else:
            pred = PredictNext(context)
            new_tone, val = ToneRandChoice(pred)
        context += (new_tone,)
        probs.append(val)
    return list(context), probs, alt

mel = GenerateMelodyWithChange(61, 8, 3, (True, False))
if not mel == False:
    tones, probs, alts = mel
    print(tones)
    print(probs)
    print(alts)
else:
    print("Error")

[61, 63, 64, 59, 66, 64, 63, 64]
[0.27910418255245845, 0.08786784252136569, 0.029477531588142578, 0.13744009562949164, 0.5355543003785811, 0.9005857112877629, 0.7950070467044044]
(63, 0.31171142432328663)


Now the same for trees

In [170]:
def PredictTwoNextWithAlternative(seq, surprise):
    probs = defaultdict(float)
    remaining_prob = 1.0
    order = len(seq)

    for o in range(order, 0, -1):
        context = seq[order - o :]
        total = sum(
            n_gram_counts[o][context + (s,)] for s in individual_tones
        )
        if total == 0:
            continue
        distinct = sum(
            1 for s in individual_tones if n_gram_counts[o][context + (s,)] > 0
        )
        p_esc = distinct / (total + distinct)

        for s in individual_tones:
            count = n_gram_counts[o][context + (s,)]
            if count > 0:
                probs[s] += remaining_prob * (1 - p_esc) * (count / total)
        
        remaining_prob *= p_esc

    unseen_prob = remaining_prob / len(individual_tones)
    for s in individual_tones:
        probs[s] += unseen_prob
    
    probs.pop(seq[-1], None)

    surprisal1, surprisal2 = surprise

    if surprisal1:
        min_val1, max_val1 = 0.01, 0.05 
    else:
        min_val1, max_val1 = 0.15, 1
    
    if surprisal2:
        min_val2, max_val2 = 0.01, 0.05 
    else:
        min_val2, max_val2 = 0.15, 1

    keys = list(probs.keys())
    values = list(probs.values())

    picked_real1, picked_real2, picked_alt = False, False, False

    real1, real2, alt = False, False, False
    i,j, k = 0,0,0

    while not picked_real1 and i < 100:
        selected_key = random.choices(keys, weights=values, k=1)[0]
        selected_value = probs[selected_key]

        if min_val1 <= selected_value <= max_val1:
            real1 = selected_key, selected_value
            probs.pop(selected_key, None)
            picked_real1 = True
        
        i += 1
    
    while not picked_real2 and k < 100:
        selected_key = random.choices(keys, weights=values, k=1)[0]
        selected_value = probs[selected_key]

        if min_val2 <= selected_value <= max_val2:
            real2 = selected_key, selected_value
            probs.pop(selected_key, None)
            picked_real2 = True
        
        k += 1

    while not picked_alt and j < 100:
        selected_key = random.choices(keys, weights=values, k=1)[0]
        selected_value = probs[selected_key]

        if min_val2 <= selected_value <= max_val2:
            alt = selected_key, selected_value
            picked_alt = True

        j += 1
    
    return real1, real2, alt

def GenerateBinaryTreeWithChange(probe, length, pos, surprise):
    tree = [0 for _ in range(2**length-1)]
    prob_tree = [0 for _ in range(2**length-1)]
    alts = []

    tree[0] = probe
    prob_tree[0] = 1
    for layer in range(1,length,1):
        if layer+1 == pos:
            fst_pos = 2**layer - 1
            for node in range(0,2**layer, 2):
                parents = [math.ceil((node+fst_pos)/2)-1]
                for i in range(layer-2):
                    parents.append(math.ceil(parents[-1]/2) - 1)
                context = tuple([tree[i] for i in parents])
                real1, real2, alt = PredictTwoNextWithAlternative(context, surprise)
                if real1 == False or real2 == False or alt == False:
                    return False
                tone1, prob1 = real1
                tone2, prob2 = real2
                tree[fst_pos + node] = tone1
                tree[fst_pos + node + 1] = tone2
                alts.append(alt)

                prob_tree[fst_pos + node] = prob1
                prob_tree[fst_pos + node + 1] = prob2
        else:
            fst_pos = 2**layer - 1
            for node in range(0,2**layer, 2):
                parents = [math.ceil((node+fst_pos)/2)-1]
                for i in range(layer-2):
                    parents.append(math.ceil(parents[-1]/2) - 1)
                context = tuple([tree[i] for i in parents])
                pred = PredictNext(context)
                tone1, prob1 = ToneRandChoice(pred)
                pred.pop(tone1)
                tone2, prob2 = ToneRandChoice(pred)
                choice = np.random.random()
                if choice > 0.5:
                    tone1, tone2 = tone2, tone1
                    prob1, prob2 = prob2, prob1
                tree[fst_pos + node] = tone1
                tree[fst_pos + node + 1] = tone2

                prob_tree[fst_pos + node] = prob1
                prob_tree[fst_pos + node + 1] = prob2
    return tree, prob_tree, alts

mel = GenerateBinaryTreeWithChange(61, 5, 2, (True, False))
if not mel == False:
    tones, probs, alts = mel
    print(tones)
    print(probs)
    print(alts)
else:
    print("Error")

[61, 70, 59, 69, 73, 57, 66, 72, 69, 77, 73, 62, 61, 62, 56, 69, 72, 67, 69, 75, 68, 73, 68, 57, 60, 62, 61, 61, 74, 56, 61]
[1, 0.02312406450337485, 0.1734813183089045, 0.05208457977768682, 0.09321724344239066, 0.13996980231047437, 0.08589486990413989, 0.308818825265682, 0.39204012961634965, 0.04883791400071275, 0.3042843322632031, 0.1745304215386776, 0.12719795726675795, 0.09210647961306341, 0.10369730928037049, 0.304477669932297, 0.2897915573998262, 0.07991338709457015, 0.39204012961634965, 0.10272573455500754, 0.21967935929988044, 0.3042843322632031, 0.33208918818548194, 0.20298635122611847, 0.24526512397507902, 0.6686326053846695, 0.1567994893166895, 0.06368681906747548, 0.29134185453578687, 0.8207394618560742, 0.0297205155648219]
[(63, 0.27910418255245845)]


The actual generation of sequences

In [169]:
import pandas as pd

sur_factors = [
    (True, True),
    (True, False),
    (False, True),
    (False, False)
]

pos_factors = [
    (2,3),
    (4,5),
    (6,7)
]

change_factors = [
    True,
    False
]

gen_factors = [
    True,
    False
]

probe_tones = [
    60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74
]

length = 8

data = {
    "Generated": [],
    "Change": [],
    "Position": [],
    "Surprisal": [],
    "Probe": [],
    "Sequence": [],
    "Probabilites": [],
    "Alternatives": []
}

for gen in gen_factors:
    for change in change_factors:
        for pos in pos_factors:
            for surprise in sur_factors:
                probe = random.choice(probe_tones)
                if change:
                    position = random.choice(pos)
                    if gen:
                        mel = GenerateBinaryTreeWithChange(probe, length, position, surprise)
                        while mel == False:
                            mel = GenerateBinaryTreeWithChange(probe, length, position, surprise)
                        seq, probs, alt = mel
                    else:
                        mel = GenerateMelodyWithChange(probe, length, position, surprise)
                        while mel == False:
                            mel = GenerateMelodyWithChange(probe, length, position, surprise)
                        seq, probs, alt = mel
                else:
                    position = -1
                    surprise = False
                    if gen:
                        seq, probs = GenerateBinaryTree(probe, length)
                        alt = False
                    else:
                        mel = GenerateMelody(probe, length)
                        seq, probs = mel
                        alt = False

                data["Generated"].append(gen)
                data["Change"].append(change)
                data["Position"].append(position)
                data["Surprisal"].append(surprise)
                data["Probe"].append(probe)
                data["Sequence"].append(seq)
                data["Probabilites"].append(probs)
                data["Alternatives"].append(alt)

df = pd.DataFrame(data)
df.to_csv("sequences.csv", index=False)

KeyboardInterrupt: 

In [140]:

seq = [
    60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74
]

print(ConvertToFreqs(seq))

[261.626, 277.183, 293.665, 311.127, 329.628, 349.228, 369.994, 391.995, 415.305, 440.0, 466.164, 493.883, 523.251, 554.365, 587.33]
